# Gap-CAM for MSCOCO / ViT-B-32

This notebook implements the first-pass **Gap-CAM** pipeline requested in `gradcam_impl_instructions.md` for:

- dataset: `MSCOCO`
- model: `ViT-B-32`
- pretrained weights: `laion2b_s34b_b79k`
- alignment width: `d_sub=256`
- bad-dimension selection: strong bad dimensions with top-`K=32`

The notebook uses the repo's precomputed MSCOCO embeddings to:

1. compute dataset-level gap / semantic statistics on the train split,
2. rank validation pairs by the signed bad target,
3. generate bad / good / difference heatmaps with ViT patch tokens,
4. save per-sample outputs plus aggregate entropy summaries.

Assumption for this first pass: dataset-level statistics are computed from **train2017** embeddings, while qualitative ranking and rendering are done on **val2017** pairs.


In [1]:
import importlib.util
import json
import math
import os
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from PIL import Image
from tqdm.auto import tqdm

try:
    from torchvision import transforms as T
except ImportError:
    T = None

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
REPO_ROOT = Path("/mnt/media/emanuele/few_dimensions")
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

print(f"Using device: {DEVICE}")
print(f"Repo root   : {REPO_ROOT}")


/opt/anaconda3/envs/few_dim_modalitygap/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Repo root   : /mnt/media/emanuele/few_dimensions


In [2]:
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"
MODEL_TAG = f"{MODEL_NAME}___{PRETRAINED}"

DATA_ROOT = REPO_ROOT / "dataset" / "mscoco" / "data" / "mscoco"
PRECOMP_TRAIN_DIR = DATA_ROOT / MODEL_TAG / "precomputed_train2017_clip_imagenet"
PRECOMP_VAL_DIR = DATA_ROOT / MODEL_TAG / "precomputed_val2017_clip_imagenet"
COCO_TRAIN_IMAGE_DIR = DATA_ROOT / "train2017"
COCO_VAL_IMAGE_DIR = DATA_ROOT / "val2017"
CAPTIONS_VAL_JSON = DATA_ROOT / "annotations" / "captions_val2017.json"
RESULTS_JSON = REPO_ROOT / "comparison" / "results" / "few_dimensions_max_retrieval_results.json"

OUTPUT_ROOT = REPO_ROOT / "analysis_2" / "gapcam_outputs" / "mscoco_vitb32_laion2b_s34b_b79k"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR = OUTPUT_ROOT / "samples"
AGGREGATES_DIR = OUTPUT_ROOT / "aggregates"
CACHE_DIR = OUTPUT_ROOT / "cache"
for _path in (SAMPLES_DIR, AGGREGATES_DIR, CACHE_DIR):
    _path.mkdir(parents=True, exist_ok=True)

D_SUB = 256
TOPK_BAD = 32
N_FIT = 10_000
MAX_STATS_SAMPLES = None
MAX_VAL_RANK_SAMPLES = None
NUM_TOP_SAMPLES = 24
NUM_RENDER_SAMPLES = 8
MAX_PER_LABEL = 2
FORCE_RECOMPUTE_STATS = False
FORCE_ABS_BAD_TARGET = False
AUTO_BAD_TARGET_FALLBACK = True
BAD_TARGET_ABS_SCORE_RATIO = 0.25
FORCE_POSITIVE_GOOD_TARGET = False
AUTO_GOOD_TARGET_FALLBACK = True
GOOD_TARGET_POS_SCORE_RATIO = 0.25
VIT_BLOCK_INDEX = -2
EPS = 1e-8

with open(RESULTS_JSON, "r") as f:
    retrieval_results = json.load(f)

best_params = retrieval_results[MODEL_TAG][str(D_SUB)]["mscoco_imagenet"]["best_params"]
TAU_GAP = float(best_params["gap_thr"])
TAU_IMP = float(best_params["imp_thr"])

print(f"tau_gap={TAU_GAP:.6f} | tau_imp={TAU_IMP:.6f}")

required_paths = [
    PRECOMP_TRAIN_DIR,
    PRECOMP_VAL_DIR,
    COCO_VAL_IMAGE_DIR,
    CAPTIONS_VAL_JSON,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required paths:\n- " + "\n- ".join(missing_paths))


tau_gap=0.761949 | tau_imp=0.208490


In [3]:
def iter_npz_shards(precomputed_dir: Path) -> Iterable[Path]:
    for shard_path in sorted(precomputed_dir.glob("*.npz")):
        yield shard_path

def to_float_tensor(array: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(np.asarray(array)).float()

def l2_normalize(x: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    return x / (x.norm(dim=-1, keepdim=True) + eps)

def coerce_text_embeddings(raw_text_emb: np.ndarray) -> np.ndarray:
    if raw_text_emb.dtype != object and raw_text_emb.ndim == 2:
        return raw_text_emb.astype(np.float32)
    if raw_text_emb.dtype != object and raw_text_emb.ndim == 3:
        return raw_text_emb[:, 0, :].astype(np.float32)
    if raw_text_emb.dtype == object:
        rows = []
        for item in raw_text_emb:
            arr = np.asarray(item, dtype=np.float32)
            if arr.ndim == 1:
                rows.append(arr)
            else:
                rows.append(arr[0])
        return np.stack(rows, axis=0).astype(np.float32)
    raise ValueError(f"Unsupported text embedding format: dtype={raw_text_emb.dtype}, ndim={raw_text_emb.ndim}")

def load_caption_metadata(captions_json: Path) -> Tuple[Dict[int, dict], Dict[int, dict], Dict[int, List[dict]]]:
    with open(captions_json, "r") as f:
        payload = json.load(f)
    image_lookup = {int(item["id"]): item for item in payload["images"]}
    caption_lookup = {int(item["id"]): item for item in payload["annotations"]}
    captions_by_image = defaultdict(list)
    for ann in payload["annotations"]:
        captions_by_image[int(ann["image_id"])].append(ann)
    return image_lookup, caption_lookup, dict(captions_by_image)

IMAGE_LOOKUP, CAPTION_LOOKUP, CAPTIONS_BY_IMAGE = load_caption_metadata(CAPTIONS_VAL_JSON)

def resolve_caption_text(img_id: int, caption_id: int) -> str:
    ann = CAPTION_LOOKUP.get(int(caption_id))
    if ann is not None:
        return str(ann["caption"])
    img_anns = CAPTIONS_BY_IMAGE.get(int(img_id), [])
    if img_anns:
        return str(img_anns[0]["caption"])
    return ""

def resolve_file_name(img_id: int, fallback: Optional[str] = None) -> str:
    info = IMAGE_LOOKUP.get(int(img_id))
    if info is not None:
        return str(info["file_name"])
    if fallback is not None:
        return str(fallback)
    return f"{int(img_id):012d}.jpg"

def load_val_image(file_name: str) -> Image.Image:
    image_path = COCO_VAL_IMAGE_DIR / file_name
    with Image.open(image_path) as img:
        return img.convert("RGB")

def slugify(text: str) -> str:
    clean = []
    for ch in str(text).lower():
        if ch.isalnum():
            clean.append(ch)
        elif ch in {" ", "-", "_", ","}:
            clean.append("-")
    slug = "".join(clean).strip("-")
    while "--" in slug:
        slug = slug.replace("--", "-")
    return slug or "item"


In [4]:
def compute_modality_means_from_shards(precomputed_dir: Path, max_samples: Optional[int] = None) -> Tuple[torch.Tensor, torch.Tensor, int]:
    sum_img = None
    sum_txt = None
    count = 0

    for shard_path in tqdm(list(iter_npz_shards(precomputed_dir)), desc=f"Means from {precomputed_dir.name}"):
        shard = np.load(shard_path, allow_pickle=True)
        txt = l2_normalize(to_float_tensor(coerce_text_embeddings(shard["text_emb"])))
        img = l2_normalize(to_float_tensor(shard["vision_emb"]))

        if max_samples is not None:
            remaining = max_samples - count
            if remaining <= 0:
                break
            txt = txt[:remaining]
            img = img[:remaining]

        txt_sum = txt.sum(dim=0, dtype=torch.float64)
        img_sum = img.sum(dim=0, dtype=torch.float64)
        sum_txt = txt_sum if sum_txt is None else (sum_txt + txt_sum)
        sum_img = img_sum if sum_img is None else (sum_img + img_sum)
        count += int(txt.shape[0])

        if max_samples is not None and count >= max_samples:
            break

    mu_img = (sum_img / count).float()
    mu_txt = (sum_txt / count).float()
    return mu_img, mu_txt, count

def collect_fit_embeddings(precomputed_dir: Path, n_fit: int) -> Tuple[torch.Tensor, torch.Tensor]:
    img_batches = []
    txt_batches = []
    count = 0

    for shard_path in tqdm(list(iter_npz_shards(precomputed_dir)), desc=f"Collect fit embeddings from {precomputed_dir.name}"):
        shard = np.load(shard_path, allow_pickle=True)
        txt = l2_normalize(to_float_tensor(coerce_text_embeddings(shard["text_emb"])))
        img = l2_normalize(to_float_tensor(shard["vision_emb"]))

        remaining = n_fit - count
        if remaining <= 0:
            break
        txt = txt[:remaining]
        img = img[:remaining]

        txt_batches.append(txt)
        img_batches.append(img)
        count += int(txt.shape[0])
        if count >= n_fit:
            break

    return torch.cat(img_batches, dim=0), torch.cat(txt_batches, dim=0)

def fit_subspace_alignment_from_embeddings(X_img: torch.Tensor, Y_txt: torch.Tensor, d_sub: int = D_SUB) -> Dict[str, torch.Tensor]:
    mu_x = X_img.mean(dim=0, keepdim=True)
    mu_y = Y_txt.mean(dim=0, keepdim=True)

    Xc = (X_img - mu_x).cpu().numpy()
    Yc = (Y_txt - mu_y).cpu().numpy()

    _, _, vtx = np.linalg.svd(Xc, full_matrices=False)
    _, _, vty = np.linalg.svd(Yc, full_matrices=False)

    W_X = torch.from_numpy(vtx[:d_sub].T).float()
    W_Y = torch.from_numpy(vty[:d_sub].T).float()
    Phi = W_Y.T @ W_X

    return {
        "mu_X_fit": mu_x.float().squeeze(0),
        "mu_Y_fit": mu_y.float().squeeze(0),
        "W_X": W_X,
        "W_Y": W_Y,
        "Phi": Phi.float(),
        "d_sub": int(d_sub),
    }

def minmax_normalize(v: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    return (v - v.min()) / (v.max() - v.min() + eps)

def compute_gap_scores(mu_img: torch.Tensor, mu_txt: torch.Tensor) -> torch.Tensor:
    return (mu_img - mu_txt).abs()

def compute_semantic_scores(W_X: torch.Tensor, W_Y: torch.Tensor) -> torch.Tensor:
    s_img = torch.sum(W_X ** 2, dim=1)
    s_txt = torch.sum(W_Y ** 2, dim=1)
    return 0.5 * (s_img + s_txt), s_img, s_txt

def build_dimension_sets(
    g_tilde: torch.Tensor,
    s_tilde: torch.Tensor,
    tau_gap: float,
    tau_imp: float,
    topk_bad: Optional[int] = TOPK_BAD,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    good_mask = (s_tilde >= tau_imp) & (g_tilde <= tau_gap)
    bad_mask = (s_tilde < tau_imp) & (g_tilde > tau_gap)
    good_indices = torch.where(good_mask)[0]
    bad_indices = torch.where(bad_mask)[0]

    if bad_indices.numel() == 0:
        bad_indices = torch.where(~good_mask)[0]

    badness = g_tilde * (1.0 - s_tilde)
    if topk_bad is not None and bad_indices.numel() > topk_bad:
        keep = torch.argsort(badness[bad_indices], descending=True)[:topk_bad]
        bad_indices = bad_indices[keep]

    goodness = (1.0 - g_tilde) * s_tilde
    bad_weights = badness[bad_indices]
    good_weights = goodness[good_indices]

    bad_weights = bad_weights / (bad_weights.sum() + EPS)
    good_weights = good_weights / (good_weights.sum() + EPS)
    return good_indices, bad_indices, good_weights, bad_weights

def compute_bad_target(z_img: torch.Tensor, z_txt: torch.Tensor, stats: Dict[str, torch.Tensor]) -> torch.Tensor:
    delta = stats["delta"].to(z_img.device)
    bad_indices = stats["bad_indices"].to(z_img.device)
    bad_weights = stats["bad_weights"].to(z_img.device)
    signed_gap = torch.sign(delta[bad_indices]) * (z_img[bad_indices] - z_txt[bad_indices])
    return torch.sum(bad_weights * F.relu(signed_gap))

def compute_bad_target_abs(z_img: torch.Tensor, z_txt: torch.Tensor, stats: Dict[str, torch.Tensor]) -> torch.Tensor:
    bad_indices = stats["bad_indices"].to(z_img.device)
    bad_weights = stats["bad_weights"].to(z_img.device)
    return torch.sum(bad_weights * (z_img[bad_indices] - z_txt[bad_indices]).abs())

def compute_good_target(z_img: torch.Tensor, z_txt: torch.Tensor, stats: Dict[str, torch.Tensor]) -> torch.Tensor:
    good_indices = stats["good_indices"].to(z_img.device)
    good_weights = stats["good_weights"].to(z_img.device)
    return torch.sum(good_weights * z_img[good_indices] * z_txt[good_indices])

def compute_good_target_positive(z_img: torch.Tensor, z_txt: torch.Tensor, stats: Dict[str, torch.Tensor]) -> torch.Tensor:
    good_indices = stats["good_indices"].to(z_img.device)
    good_weights = stats["good_weights"].to(z_img.device)
    return torch.sum(good_weights * F.relu(z_img[good_indices] * z_txt[good_indices]))

def summarize_top_dimensions(z_img: torch.Tensor, z_txt: torch.Tensor, stats: Dict[str, torch.Tensor], topk: int = 8) -> Dict[str, List[dict]]:
    delta = stats["delta"]
    bad_indices = stats["bad_indices"]
    good_indices = stats["good_indices"]
    bad_weights = stats["bad_weights"]
    good_weights = stats["good_weights"]

    bad_scores = bad_weights * F.relu(torch.sign(delta[bad_indices]) * (z_img[bad_indices] - z_txt[bad_indices]))
    good_scores = good_weights * (z_img[good_indices] * z_txt[good_indices])

    top_bad_pos = torch.argsort(bad_scores, descending=True)[:topk]
    top_good_pos = torch.argsort(good_scores, descending=True)[:topk]

    top_bad = [
        {"dim": int(bad_indices[idx]), "score": float(bad_scores[idx].item())}
        for idx in top_bad_pos
    ]
    top_good = [
        {"dim": int(good_indices[idx]), "score": float(good_scores[idx].item())}
        for idx in top_good_pos
    ]
    return {"top_bad_dims": top_bad, "top_good_dims": top_good}


In [5]:
STATS_CACHE_PATH = CACHE_DIR / f"dimension_stats_dsub{D_SUB}_topk{TOPK_BAD}.pt"

def compute_or_load_dimension_stats(force_recompute: bool = FORCE_RECOMPUTE_STATS) -> Dict[str, torch.Tensor]:
    if STATS_CACHE_PATH.exists() and not force_recompute:
        stats = torch.load(STATS_CACHE_PATH, map_location="cpu")
        print(f"Loaded cached stats from {STATS_CACHE_PATH}")
        return stats

    mu_img, mu_txt, mean_count = compute_modality_means_from_shards(
        PRECOMP_TRAIN_DIR,
        max_samples=MAX_STATS_SAMPLES,
    )
    g = compute_gap_scores(mu_img, mu_txt)
    g_tilde = minmax_normalize(g)

    X_fit_img, Y_fit_txt = collect_fit_embeddings(PRECOMP_TRAIN_DIR, n_fit=N_FIT)
    subspace = fit_subspace_alignment_from_embeddings(X_fit_img, Y_fit_txt, d_sub=D_SUB)
    s, s_img, s_txt = compute_semantic_scores(subspace["W_X"], subspace["W_Y"])
    s_tilde = minmax_normalize(s)

    good_indices, bad_indices, good_weights, bad_weights = build_dimension_sets(
        g_tilde=g_tilde,
        s_tilde=s_tilde,
        tau_gap=TAU_GAP,
        tau_imp=TAU_IMP,
        topk_bad=TOPK_BAD,
    )

    stats = {
        "mu_X": mu_img,
        "mu_Y": mu_txt,
        "delta": mu_img - mu_txt,
        "g": g,
        "g_tilde": g_tilde,
        "s": s,
        "s_img": s_img,
        "s_txt": s_txt,
        "s_tilde": s_tilde,
        "W_X": subspace["W_X"],
        "W_Y": subspace["W_Y"],
        "Phi": subspace["Phi"],
        "good_indices": good_indices,
        "bad_indices": bad_indices,
        "good_weights": good_weights,
        "bad_weights": bad_weights,
        "tau_gap": float(TAU_GAP),
        "tau_imp": float(TAU_IMP),
        "d_sub": int(D_SUB),
        "topk_bad": int(TOPK_BAD),
        "mean_count": int(mean_count),
        "fit_count": int(X_fit_img.shape[0]),
        "stats_split": "train2017",
        "model_name": MODEL_NAME,
        "pretrained": PRETRAINED,
    }

    torch.save(stats, STATS_CACHE_PATH)
    print(f"Saved stats to {STATS_CACHE_PATH}")
    print(f"good dims={good_indices.numel()} | bad dims={bad_indices.numel()} | mean_count={mean_count}")
    return stats


In [6]:
def load_val_embeddings_with_metadata(max_samples: Optional[int] = MAX_VAL_RANK_SAMPLES) -> Dict[str, object]:
    img_batches = []
    txt_batches = []
    records = []
    count = 0

    for shard_path in tqdm(list(iter_npz_shards(PRECOMP_VAL_DIR)), desc=f"Load val embeddings from {PRECOMP_VAL_DIR.name}"):
        shard = np.load(shard_path, allow_pickle=True)
        txt_np = coerce_text_embeddings(shard["text_emb"])
        img_np = np.asarray(shard["vision_emb"], dtype=np.float32)

        txt = l2_normalize(to_float_tensor(txt_np))
        img = l2_normalize(to_float_tensor(img_np))

        img_ids = shard["img_ids"].astype(np.int64)
        caption_ids = shard["caption_ids"].astype(np.int64)
        label_ids = shard["label_ids"].astype(np.int64)
        label_names = shard["label_names"]
        fns = shard["fns"] if "fns" in shard.files else np.array([None] * len(img_ids), dtype=object)

        for row_idx in range(len(img_ids)):
            if max_samples is not None and count >= max_samples:
                break
            img_id = int(img_ids[row_idx])
            caption_id = int(caption_ids[row_idx])
            file_name = resolve_file_name(img_id, fallback=fns[row_idx])
            records.append(
                {
                    "row_index": count,
                    "img_id": img_id,
                    "caption_id": caption_id,
                    "label_id": int(label_ids[row_idx]),
                    "label_name": str(label_names[row_idx]),
                    "file_name": file_name,
                    "caption": resolve_caption_text(img_id, caption_id),
                }
            )
            count += 1

        take_n = len(records) - sum(batch.shape[0] for batch in img_batches)
        if take_n > 0:
            img_batches.append(img[:take_n])
            txt_batches.append(txt[:take_n])

        if max_samples is not None and count >= max_samples:
            break

    table = pd.DataFrame.from_records(records)
    img_embeddings = torch.cat(img_batches, dim=0)
    txt_embeddings = torch.cat(txt_batches, dim=0)
    return {
        "table": table,
        "img_embeddings": img_embeddings,
        "txt_embeddings": txt_embeddings,
    }

def score_embeddings(img_embeddings: torch.Tensor, txt_embeddings: torch.Tensor, stats: Dict[str, torch.Tensor]) -> pd.DataFrame:
    bad_idx = stats["bad_indices"]
    good_idx = stats["good_indices"]
    delta = stats["delta"]
    bad_weights = stats["bad_weights"]
    good_weights = stats["good_weights"]

    signed_gap = torch.sign(delta[bad_idx]).unsqueeze(0) * (img_embeddings[:, bad_idx] - txt_embeddings[:, bad_idx])
    bad_scores = (F.relu(signed_gap) * bad_weights.unsqueeze(0)).sum(dim=1)
    abs_bad_scores = ((img_embeddings[:, bad_idx] - txt_embeddings[:, bad_idx]).abs() * bad_weights.unsqueeze(0)).sum(dim=1)
    good_scores = ((img_embeddings[:, good_idx] * txt_embeddings[:, good_idx]) * good_weights.unsqueeze(0)).sum(dim=1)

    return pd.DataFrame(
        {
            "bad_score": bad_scores.cpu().numpy(),
            "bad_score_abs": abs_bad_scores.cpu().numpy(),
            "good_score": good_scores.cpu().numpy(),
            "bad_minus_good": (bad_scores - good_scores).cpu().numpy(),
        }
    )

def rank_val_samples(stats: Dict[str, torch.Tensor], max_samples: Optional[int] = MAX_VAL_RANK_SAMPLES) -> Dict[str, object]:
    bundle = load_val_embeddings_with_metadata(max_samples=max_samples)
    score_df = score_embeddings(bundle["img_embeddings"], bundle["txt_embeddings"], stats)
    ranked = pd.concat([bundle["table"].reset_index(drop=True), score_df], axis=1)
    ranked = ranked.sort_values(["bad_score", "bad_minus_good"], ascending=[False, False]).reset_index(drop=True)
    return {
        "table": ranked,
        "img_embeddings": bundle["img_embeddings"],
        "txt_embeddings": bundle["txt_embeddings"],
    }

def select_diverse_top_samples(ranked_df: pd.DataFrame, n_samples: int = NUM_RENDER_SAMPLES, max_per_label: int = MAX_PER_LABEL) -> pd.DataFrame:
    chosen_rows = []
    per_label = defaultdict(int)
    for _, row in ranked_df.iterrows():
        label_name = row["label_name"]
        if per_label[label_name] >= max_per_label:
            continue
        chosen_rows.append(row)
        per_label[label_name] += 1
        if len(chosen_rows) >= n_samples:
            break
    return pd.DataFrame(chosen_rows).reset_index(drop=True)


In [7]:
OPEN_CLIP_AVAILABLE = importlib.util.find_spec("open_clip") is not None
if OPEN_CLIP_AVAILABLE:
    import open_clip
else:
    open_clip = None
    print("open_clip is not installed in this kernel. Stats and ranking cells still work; rendering cells need open_clip.")

def build_openclip_model(model_name: str = MODEL_NAME, pretrained: str = PRETRAINED, device: torch.device = DEVICE):
    if open_clip is None:
        raise ImportError("Install open_clip in the notebook kernel before running the rendering cells.")
    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    model = model.to(device).eval()
    for param in model.parameters():
        param.requires_grad_(False)
    tokenizer = open_clip.get_tokenizer(model_name)
    return model, preprocess, tokenizer

def build_display_transform(preprocess):
    if T is None or not hasattr(preprocess, "transforms"):
        return None
    transforms = []
    for tr in preprocess.transforms:
        if tr.__class__.__name__ == "Normalize":
            break
        transforms.append(tr)
    if not transforms:
        return None
    return T.Compose(transforms)

def preprocess_for_model_and_display(image_pil: Image.Image, preprocess):
    model_tensor = preprocess(image_pil).unsqueeze(0)
    display_transform = build_display_transform(preprocess)
    if display_transform is not None:
        display_tensor = display_transform(image_pil)
        if isinstance(display_tensor, Image.Image):
            display_pil = display_tensor.convert("RGB")
        else:
            display_pil = T.ToPILImage()(display_tensor.clamp(0, 1))
    else:
        display_pil = image_pil.copy()
    return model_tensor, display_pil

def resolve_vit_hook_module(model, block_index: int = VIT_BLOCK_INDEX) -> Tuple[str, torch.nn.Module]:
    visual = model.visual
    candidates = []

    if hasattr(visual, "trunk") and hasattr(visual.trunk, "blocks"):
        blocks = visual.trunk.blocks
        idx = block_index if len(blocks) >= abs(block_index) else -1
        return f"visual.trunk.blocks[{idx}]", blocks[idx]
    if hasattr(visual, "transformer") and hasattr(visual.transformer, "resblocks"):
        blocks = visual.transformer.resblocks
        idx = block_index if len(blocks) >= abs(block_index) else -1
        return f"visual.transformer.resblocks[{idx}]", blocks[idx]
    if hasattr(visual, "transformer") and hasattr(visual.transformer, "blocks"):
        blocks = visual.transformer.blocks
        idx = block_index if len(blocks) >= abs(block_index) else -1
        return f"visual.transformer.blocks[{idx}]", blocks[idx]

    for name, module in visual.named_modules():
        if any(tok in name for tok in ("resblocks", "blocks")):
            candidates.append((name, module))

    if candidates:
        idx = block_index if len(candidates) >= abs(block_index) else -1
        return candidates[idx]
    raise RuntimeError("Could not find a ViT block suitable for patch-token hooking.")

def _unwrap_tensor(output):
    if torch.is_tensor(output):
        return output
    if isinstance(output, (list, tuple)):
        for item in output:
            if torch.is_tensor(item):
                return item
    raise TypeError(f"Hook output does not contain a tensor: {type(output)}")

def _canonicalize_token_tensor(tokens: torch.Tensor, batch_size: int = 1) -> torch.Tensor:
    if tokens.ndim != 3:
        raise ValueError(f"Expected 3D token tensor, got shape={tuple(tokens.shape)}")
    if tokens.shape[0] == batch_size:
        return tokens
    if tokens.shape[1] == batch_size:
        return tokens.permute(1, 0, 2)
    raise ValueError(f"Could not canonicalize token tensor with shape={tuple(tokens.shape)}")

def _infer_patch_grid(model, token_count: int) -> Tuple[int, int, int]:
    grid_size = getattr(model.visual, "grid_size", None)
    if grid_size is not None:
        if isinstance(grid_size, int):
            gh = gw = int(grid_size)
        else:
            gh, gw = int(grid_size[0]), int(grid_size[1])
        patch_tokens = gh * gw
        prefix_tokens = token_count - patch_tokens
        if prefix_tokens < 0:
            raise ValueError(f"Token count {token_count} is smaller than expected patch grid {patch_tokens}")
        return gh, gw, prefix_tokens

    prefix_tokens = 1
    patch_tokens = token_count - prefix_tokens
    side = int(round(math.sqrt(patch_tokens)))
    if side * side != patch_tokens:
        raise ValueError(f"Could not infer square patch grid from token_count={token_count}")
    return side, side, prefix_tokens

def minmax_heatmap(heatmap: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    heatmap = heatmap.float()
    return (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + eps)

def compute_vit_heatmap_from_target(model, image_tensor: torch.Tensor, text_tokens: torch.Tensor, target_fn, hook_module, saliency_mode: str = "relu") -> Tuple[np.ndarray, float, torch.Tensor, torch.Tensor, Dict[str, float]]:
    store = {}

    def forward_hook(_module, _inputs, output):
        tokens = _unwrap_tensor(output)
        store["activations"] = tokens
        if tokens.requires_grad:
            tokens.retain_grad()
            tokens.register_hook(lambda grad: store.__setitem__("gradients", grad))

    handle = hook_module.register_forward_hook(forward_hook)
    try:
        model.zero_grad(set_to_none=True)
        image_tensor = image_tensor.to(DEVICE).requires_grad_(True)
        text_tokens = text_tokens.to(DEVICE)

        img_features = model.encode_image(image_tensor)
        txt_features = model.encode_text(text_tokens)
        z_img = F.normalize(img_features, dim=-1).squeeze(0)
        z_txt = F.normalize(txt_features, dim=-1).squeeze(0)
        target = target_fn(z_img, z_txt)
        target.backward()

        activations = _canonicalize_token_tensor(store["activations"], batch_size=1)[0].detach()
        gradients = _canonicalize_token_tensor(store["gradients"], batch_size=1)[0].detach()
        gh, gw, prefix_tokens = _infer_patch_grid(model, activations.shape[0])
        activations = activations[prefix_tokens:prefix_tokens + gh * gw]
        gradients = gradients[prefix_tokens:prefix_tokens + gh * gw]

        patch_scores_signed = (activations * gradients).sum(dim=-1)
        if saliency_mode == "relu":
            patch_scores = torch.relu(patch_scores_signed)
        elif saliency_mode == "abs":
            patch_scores = patch_scores_signed.abs()
        else:
            raise ValueError(f"Unsupported saliency_mode={saliency_mode}")
        raw_stats = {
            "raw_patch_max": float(patch_scores.max().item()),
            "raw_patch_sum": float(patch_scores.sum().item()),
            "raw_patch_mean": float(patch_scores.mean().item()),
            "raw_signed_patch_max": float(patch_scores_signed.max().item()),
            "raw_signed_patch_min": float(patch_scores_signed.min().item()),
        }
        patch_grid = patch_scores.reshape(gh, gw)
        patch_grid = minmax_heatmap(patch_grid)
        return patch_grid.cpu().numpy(), float(target.item()), z_img.detach().cpu(), z_txt.detach().cpu(), raw_stats
    finally:
        handle.remove()

def upsample_heatmap(patch_map: np.ndarray, out_size: Tuple[int, int]) -> np.ndarray:
    heatmap = torch.from_numpy(np.asarray(patch_map, dtype=np.float32))[None, None]
    heatmap = F.interpolate(heatmap, size=out_size, mode="bilinear", align_corners=False)
    return heatmap[0, 0].cpu().numpy()

def make_overlay(image_pil: Image.Image, heatmap: np.ndarray, cmap: str = "inferno", alpha: float = 0.42, vmin: Optional[float] = 0.0, vmax: Optional[float] = 1.0) -> Image.Image:
    image = np.asarray(image_pil).astype(np.float32) / 255.0
    heatmap = np.asarray(heatmap, dtype=np.float32)
    if vmin is None:
        vmin = float(np.min(heatmap))
    if vmax is None:
        vmax = float(np.max(heatmap))
    heatmap_norm = np.clip((heatmap - vmin) / (vmax - vmin + EPS), 0.0, 1.0)
    colored = mpl.colormaps[cmap](heatmap_norm)[..., :3]
    blended = np.clip((1.0 - alpha) * image + alpha * colored, 0.0, 1.0)
    return Image.fromarray((blended * 255).astype(np.uint8))

def save_heatmap(path: Path, heatmap: np.ndarray, cmap: str, vmin: float, vmax: float) -> None:
    plt.imsave(path, np.asarray(heatmap, dtype=np.float32), cmap=cmap, vmin=vmin, vmax=vmax)

def compute_gapcam_maps_for_pair(model, preprocess, tokenizer, image_pil: Image.Image, caption: str, stats: Dict[str, torch.Tensor], hook_module, use_abs_bad_target: bool = FORCE_ABS_BAD_TARGET) -> Dict[str, object]:
    model_tensor, display_pil = preprocess_for_model_and_display(image_pil, preprocess)
    text_tokens = tokenizer([caption])

    signed_bad_patch, signed_bad_score, z_img, z_txt, signed_bad_stats = compute_vit_heatmap_from_target(
        model=model,
        image_tensor=model_tensor,
        text_tokens=text_tokens,
        target_fn=lambda zi, zt: compute_bad_target(zi, zt, stats),
        hook_module=hook_module,
        saliency_mode="relu",
    )

    bad_patch = signed_bad_patch
    bad_score = signed_bad_score
    bad_target_mode = "signed"
    signed_bad_patch_max = float(np.max(signed_bad_patch))
    signed_bad_raw_max = float(signed_bad_stats["raw_patch_max"])
    signed_bad_raw_sum = float(signed_bad_stats["raw_patch_sum"])
    abs_bad_score = None
    abs_bad_raw_max = None
    abs_bad_raw_sum = None

    signed_is_degenerate = (signed_bad_score <= EPS) or (signed_bad_raw_sum <= EPS)
    if use_abs_bad_target or AUTO_BAD_TARGET_FALLBACK or signed_is_degenerate:
        abs_bad_patch, abs_bad_score, z_img_abs, z_txt_abs, abs_bad_stats = compute_vit_heatmap_from_target(
            model=model,
            image_tensor=model_tensor,
            text_tokens=text_tokens,
            target_fn=lambda zi, zt: compute_bad_target_abs(zi, zt, stats),
            hook_module=hook_module,
            saliency_mode="relu",
        )
        abs_bad_raw_max = float(abs_bad_stats["raw_patch_max"])
        abs_bad_raw_sum = float(abs_bad_stats["raw_patch_sum"])
        abs_should_replace = (
            use_abs_bad_target
            or signed_is_degenerate
            or (
                AUTO_BAD_TARGET_FALLBACK
                and abs_bad_score > EPS
                and signed_bad_score <= BAD_TARGET_ABS_SCORE_RATIO * abs_bad_score
            )
        )
        if abs_should_replace:
            bad_patch = abs_bad_patch
            bad_score = abs_bad_score
            bad_target_mode = "abs_forced" if use_abs_bad_target else "abs_fallback"
            z_img = z_img_abs
            z_txt = z_txt_abs

    signed_good_patch, signed_good_score, _, _, signed_good_stats = compute_vit_heatmap_from_target(
        model=model,
        image_tensor=model_tensor,
        text_tokens=text_tokens,
        target_fn=lambda zi, zt: compute_good_target(zi, zt, stats),
        hook_module=hook_module,
        saliency_mode="abs",
    )

    good_patch = signed_good_patch
    good_score = signed_good_score
    good_target_mode = "signed"
    signed_good_raw_max = float(signed_good_stats["raw_patch_max"])
    signed_good_raw_sum = float(signed_good_stats["raw_patch_sum"])
    positive_good_score = None
    positive_good_raw_max = None
    positive_good_raw_sum = None

    signed_good_is_degenerate = (signed_good_raw_sum <= EPS) or (signed_good_score <= 0.0)
    if FORCE_POSITIVE_GOOD_TARGET or AUTO_GOOD_TARGET_FALLBACK or signed_good_is_degenerate:
        positive_good_patch, positive_good_score, _, _, positive_good_stats = compute_vit_heatmap_from_target(
            model=model,
            image_tensor=model_tensor,
            text_tokens=text_tokens,
            target_fn=lambda zi, zt: compute_good_target_positive(zi, zt, stats),
            hook_module=hook_module,
            saliency_mode="abs",
        )
        positive_good_raw_max = float(positive_good_stats["raw_patch_max"])
        positive_good_raw_sum = float(positive_good_stats["raw_patch_sum"])
        positive_should_replace = (
            FORCE_POSITIVE_GOOD_TARGET
            or signed_good_is_degenerate
            or (
                AUTO_GOOD_TARGET_FALLBACK
                and positive_good_score > EPS
                and signed_good_score <= GOOD_TARGET_POS_SCORE_RATIO * positive_good_score
            )
        )
        if positive_should_replace:
            good_patch = positive_good_patch
            good_score = positive_good_score
            good_target_mode = "positive_forced" if FORCE_POSITIVE_GOOD_TARGET else "positive_fallback"

    display_hw = (display_pil.height, display_pil.width)
    bad_img = upsample_heatmap(bad_patch, display_hw)
    good_img = upsample_heatmap(good_patch, display_hw)
    diff_img = bad_img - good_img
    diff_abs_max = float(max(abs(diff_img.min()), abs(diff_img.max()), EPS))

    dim_summary = summarize_top_dimensions(z_img, z_txt, stats, topk=8)
    return {
        "display_image": display_pil,
        "bad_patch": bad_patch,
        "good_patch": good_patch,
        "bad_heatmap": bad_img,
        "good_heatmap": good_img,
        "diff_heatmap": diff_img,
        "bad_score": bad_score,
        "good_score": good_score,
        "good_target_mode": good_target_mode,
        "good_saliency_mode": "abs",
        "bad_target_mode": bad_target_mode,
        "signed_bad_score": float(signed_bad_score),
        "signed_bad_patch_max": float(signed_bad_patch_max),
        "signed_bad_raw_max": float(signed_bad_raw_max),
        "signed_bad_raw_sum": float(signed_bad_raw_sum),
        "abs_bad_score": None if abs_bad_score is None else float(abs_bad_score),
        "abs_bad_raw_max": None if abs_bad_raw_max is None else float(abs_bad_raw_max),
        "abs_bad_raw_sum": None if abs_bad_raw_sum is None else float(abs_bad_raw_sum),
        "signed_good_score": float(signed_good_score),
        "signed_good_raw_max": float(signed_good_raw_max),
        "signed_good_raw_sum": float(signed_good_raw_sum),
        "positive_good_score": None if positive_good_score is None else float(positive_good_score),
        "positive_good_raw_max": None if positive_good_raw_max is None else float(positive_good_raw_max),
        "positive_good_raw_sum": None if positive_good_raw_sum is None else float(positive_good_raw_sum),
        "good_raw_max": float(max(signed_good_raw_max, positive_good_raw_max or 0.0) if good_target_mode != "signed" else signed_good_raw_max),
        "good_raw_sum": float((positive_good_raw_sum if good_target_mode != "signed" else signed_good_raw_sum) or 0.0),
        "diff_abs_max": diff_abs_max,
        "z_img": z_img,
        "z_txt": z_txt,
        **dim_summary,
    }


In [8]:
def normalize_sum1(heatmap: np.ndarray) -> np.ndarray:
    heatmap = np.asarray(heatmap, dtype=np.float32)
    heatmap = np.clip(heatmap, a_min=0.0, a_max=None)
    return heatmap / (heatmap.sum() + EPS)

def heatmap_entropy(heatmap: np.ndarray) -> float:
    probs = normalize_sum1(heatmap)
    return float(-(probs * np.log(probs + EPS)).sum())

def save_sample_outputs(sample_rank: int, sample_row: pd.Series, outputs: Dict[str, object], sample_dir: Path) -> dict:
    sample_dir.mkdir(parents=True, exist_ok=True)
    original_image = load_val_image(sample_row["file_name"])
    original_image.save(sample_dir / "original_image.png")
    outputs["display_image"].save(sample_dir / "model_view.png")

    save_heatmap(sample_dir / "bad_heatmap.png", outputs["bad_heatmap"], cmap="inferno", vmin=0.0, vmax=1.0)
    save_heatmap(sample_dir / "good_heatmap.png", outputs["good_heatmap"], cmap="viridis", vmin=0.0, vmax=1.0)
    save_heatmap(sample_dir / "diff_heatmap.png", outputs["diff_heatmap"], cmap="coolwarm", vmin=-outputs["diff_abs_max"], vmax=outputs["diff_abs_max"])

    make_overlay(outputs["display_image"], outputs["bad_heatmap"], cmap="inferno").save(sample_dir / "bad_overlay.png")
    make_overlay(outputs["display_image"], outputs["good_heatmap"], cmap="viridis").save(sample_dir / "good_overlay.png")
    make_overlay(
        outputs["display_image"],
        outputs["diff_heatmap"],
        cmap="coolwarm",
        vmin=-outputs["diff_abs_max"],
        vmax=outputs["diff_abs_max"],
    ).save(sample_dir / "diff_overlay.png")

    metadata = {
        "rank": int(sample_rank),
        "img_id": int(sample_row["img_id"]),
        "caption_id": int(sample_row["caption_id"]),
        "file_name": sample_row["file_name"],
        "caption": sample_row["caption"],
        "label_id": int(sample_row["label_id"]),
        "label_name": sample_row["label_name"],
        "bad_score": float(outputs["bad_score"]),
        "good_score": float(outputs["good_score"]),
        "good_target_mode": outputs["good_target_mode"],
        "good_saliency_mode": outputs["good_saliency_mode"],
        "bad_target_mode": outputs["bad_target_mode"],
        "signed_bad_score": float(outputs["signed_bad_score"]),
        "signed_bad_patch_max": float(outputs["signed_bad_patch_max"]),
        "signed_bad_raw_max": float(outputs["signed_bad_raw_max"]),
        "signed_bad_raw_sum": float(outputs["signed_bad_raw_sum"]),
        "abs_bad_score": None if outputs["abs_bad_score"] is None else float(outputs["abs_bad_score"]),
        "abs_bad_raw_max": None if outputs["abs_bad_raw_max"] is None else float(outputs["abs_bad_raw_max"]),
        "abs_bad_raw_sum": None if outputs["abs_bad_raw_sum"] is None else float(outputs["abs_bad_raw_sum"]),
        "good_raw_max": float(outputs["good_raw_max"]),
        "good_raw_sum": float(outputs["good_raw_sum"]),
        "signed_good_score": float(outputs["signed_good_score"]),
        "signed_good_raw_max": float(outputs["signed_good_raw_max"]),
        "signed_good_raw_sum": float(outputs["signed_good_raw_sum"]),
        "positive_good_score": None if outputs["positive_good_score"] is None else float(outputs["positive_good_score"]),
        "positive_good_raw_max": None if outputs["positive_good_raw_max"] is None else float(outputs["positive_good_raw_max"]),
        "positive_good_raw_sum": None if outputs["positive_good_raw_sum"] is None else float(outputs["positive_good_raw_sum"]),
        "bad_entropy": heatmap_entropy(outputs["bad_heatmap"]),
        "good_entropy": heatmap_entropy(outputs["good_heatmap"]),
        "top_bad_dims": outputs["top_bad_dims"],
        "top_good_dims": outputs["top_good_dims"],
    }
    with open(sample_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    return metadata

def run_gapcam_for_samples(selected_df: pd.DataFrame, stats: Dict[str, torch.Tensor], model, preprocess, tokenizer, hook_module, output_root: Path = SAMPLES_DIR) -> List[dict]:
    render_results = []
    for sample_rank, (_, row) in enumerate(tqdm(selected_df.iterrows(), total=len(selected_df), desc="Render Gap-CAM samples"), start=1):
        image = load_val_image(row["file_name"])
        outputs = compute_gapcam_maps_for_pair(
            model=model,
            preprocess=preprocess,
            tokenizer=tokenizer,
            image_pil=image,
            caption=row["caption"],
            stats=stats,
            hook_module=hook_module,
            use_abs_bad_target=FORCE_ABS_BAD_TARGET,
        )
        sample_dir = output_root / f"rank_{sample_rank:03d}_img_{int(row['img_id']):012d}_{slugify(row['label_name'])}"
        metadata = save_sample_outputs(sample_rank, row, outputs, sample_dir)
        render_results.append({
            **metadata,
            "sample_dir": str(sample_dir),
            "bad_heatmap": outputs["bad_heatmap"],
            "good_heatmap": outputs["good_heatmap"],
            "diff_heatmap": outputs["diff_heatmap"],
        })
    return render_results

def aggregate_render_results(render_results: List[dict], output_dir: Path = AGGREGATES_DIR) -> Tuple[pd.DataFrame, pd.DataFrame]:
    output_dir.mkdir(parents=True, exist_ok=True)
    per_sample_rows = []
    grouped = defaultdict(list)

    for item in render_results:
        per_sample_rows.append(
            {
                "rank": item["rank"],
                "img_id": item["img_id"],
                "label_name": item["label_name"],
                "bad_score": item["bad_score"],
                "good_score": item["good_score"],
                "good_target_mode": item["good_target_mode"],
                "bad_entropy": item["bad_entropy"],
                "good_entropy": item["good_entropy"],
                "sample_dir": item["sample_dir"],
            }
        )
        grouped[item["label_name"]].append(item)

    per_sample_df = pd.DataFrame(per_sample_rows).sort_values("bad_score", ascending=False)
    per_sample_df.to_csv(output_dir / "per_sample_summary.csv", index=False)

    category_rows = []
    for label_name, items in grouped.items():
        bad_stack = np.stack([normalize_sum1(item["bad_heatmap"]) for item in items], axis=0)
        good_stack = np.stack([normalize_sum1(item["good_heatmap"]) for item in items], axis=0)
        mean_bad = bad_stack.mean(axis=0)
        mean_good = good_stack.mean(axis=0)
        mean_diff = mean_bad - mean_good
        diff_abs_max = float(max(abs(mean_diff.min()), abs(mean_diff.max()), EPS))

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(mean_bad, cmap="inferno")
        axes[0].set_title(f"Bad mean\n{label_name}")
        axes[1].imshow(mean_good, cmap="viridis")
        axes[1].set_title("Good mean")
        im = axes[2].imshow(mean_diff, cmap="coolwarm", vmin=-diff_abs_max, vmax=diff_abs_max)
        axes[2].set_title("Bad - Good")
        for ax in axes:
            ax.axis("off")
        fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(output_dir / f"aggregate_{slugify(label_name)}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)

        category_rows.append(
            {
                "label_name": label_name,
                "n_samples": len(items),
                "mean_bad_entropy": float(np.mean([item["bad_entropy"] for item in items])),
                "mean_good_entropy": float(np.mean([item["good_entropy"] for item in items])),
            }
        )

    category_df = pd.DataFrame(category_rows).sort_values(["n_samples", "mean_bad_entropy"], ascending=[False, True])
    category_df.to_csv(output_dir / "category_summary.csv", index=False)
    return per_sample_df, category_df


In [9]:
stats = compute_or_load_dimension_stats(force_recompute=FORCE_RECOMPUTE_STATS)
print(
    f"Stats ready | good dims={stats['good_indices'].numel()} | "
    f"bad dims={stats['bad_indices'].numel()} | fit_count={stats['fit_count']}"
)

ranked_bundle = rank_val_samples(stats, max_samples=MAX_VAL_RANK_SAMPLES)
ranked_val = ranked_bundle["table"]
ranked_val.to_csv(CACHE_DIR / "ranked_val_samples.csv", index=False)

display(
    ranked_val[
        [
            "img_id",
            "caption_id",
            "label_name",
            "bad_score",
            "bad_score_abs",
            "good_score",
            "bad_minus_good",
            "caption",
        ]
    ].head(NUM_TOP_SAMPLES)
)

selected_samples = select_diverse_top_samples(
    ranked_val,
    n_samples=NUM_RENDER_SAMPLES,
    max_per_label=MAX_PER_LABEL,
)
selected_samples.to_csv(CACHE_DIR / "selected_samples.csv", index=False)
display(selected_samples[["img_id", "label_name", "bad_score", "bad_score_abs", "good_score", "caption"]])


Means from precomputed_train2017_clip_imagenet:   0%|          | 0/58 [00:00<?, ?it/s]

Means from precomputed_train2017_clip_imagenet: 100%|██████████| 58/58 [00:03<00:00, 18.72it/s]
Collect fit embeddings from precomputed_train2017_clip_imagenet:   7%|▋         | 4/58 [00:00<00:03, 15.66it/s]


Saved stats to /mnt/media/emanuele/few_dimensions/analysis_2/gapcam_outputs/mscoco_vitb32_laion2b_s34b_b79k/cache/dimension_stats_dsub256_topk32.pt
good dims=507 | bad dims=4 | mean_count=118287
Stats ready | good dims=507 | bad dims=4 | fit_count=10000


Load val embeddings from precomputed_val2017_clip_imagenet: 100%|██████████| 3/3 [00:00<00:00, 21.63it/s]


,img_id,caption_id,label_name,bad_score,bad_score_abs,good_score,bad_minus_good,caption
0,293390,595282,"washbasin, handbasin, washbowl, lavabo, wash-h...",0.433592,0.433592,0.000454,0.433138,A VERY NICE BATHROOM IN THE MIDDLE OF A REMODEL
1,534664,805642,chest,0.421999,0.421999,0.000554,0.421445,A close up photo of suitcases and luggage
2,429598,345805,"microwave, microwave oven",0.413838,0.413838,0.000557,0.413281,An interesting kitchen renovation with brick a...
3,489091,67122,"medicine chest, medicine cabinet",0.412021,0.412021,0.000453,0.411568,A residential bathroom has a modern motel look...
4,356432,534239,table lamp,0.407102,0.407102,0.000496,0.406606,Picture of living room with modern furniture a...
5,232088,453640,"home theater, home theatre",0.405908,0.405908,0.000383,0.405525,A living room filled with furniture and decor.
6,541291,804991,soap dispenser,0.396671,0.396671,0.000407,0.396264,"The old, dilapidated bathroom has fallen into..."
7,53909,603587,iPod,0.396144,0.396144,0.000353,0.395791,a picture of a cellphone on a cellphone
8,161978,414370,knee pad,0.395973,0.395973,0.000680,0.395293,Different people are doing skateboard tricks a...
9,575970,801312,"china cabinet, china closet",0.395114,0.395114,0.000367,0.394747,The living room is nicely cleaned and organized.


,img_id,label_name,bad_score,bad_score_abs,good_score,caption
0,293390,"washbasin, handbasin, washbowl, lavabo, wash-h...",0.433592,0.433592,0.000454,A VERY NICE BATHROOM IN THE MIDDLE OF A REMODEL
1,534664,chest,0.421999,0.421999,0.000554,A close up photo of suitcases and luggage
2,429598,"microwave, microwave oven",0.413838,0.413838,0.000557,An interesting kitchen renovation with brick a...
3,489091,"medicine chest, medicine cabinet",0.412021,0.412021,0.000453,A residential bathroom has a modern motel look...
4,356432,table lamp,0.407102,0.407102,0.000496,Picture of living room with modern furniture a...
5,232088,"home theater, home theatre",0.405908,0.405908,0.000383,A living room filled with furniture and decor.
6,541291,soap dispenser,0.396671,0.396671,0.000407,"The old, dilapidated bathroom has fallen into..."
7,53909,iPod,0.396144,0.396144,0.000353,a picture of a cellphone on a cellphone


In [10]:
model = preprocess = tokenizer = hook_module = hook_module_name = None

if OPEN_CLIP_AVAILABLE:
    model, preprocess, tokenizer = build_openclip_model()
    hook_module_name, hook_module = resolve_vit_hook_module(model)
    print(f"Hooking ViT tokens from: {hook_module_name}")
else:
    print("Skipping model construction because open_clip is unavailable in this kernel.")


Hooking ViT tokens from: visual.transformer.resblocks[-2]


In [11]:
if model is None or preprocess is None or tokenizer is None or hook_module is None:
    raise RuntimeError("Load open_clip and run the previous cell before rendering heatmaps.")

render_results = run_gapcam_for_samples(
    selected_df=selected_samples,
    stats=stats,
    model=model,
    preprocess=preprocess,
    tokenizer=tokenizer,
    hook_module=hook_module,
    output_root=SAMPLES_DIR,
)

render_summary = pd.DataFrame(
    [
        {
            "rank": item["rank"],
            "label_name": item["label_name"],
            "bad_score": item["bad_score"],
            "good_score": item["good_score"],
            "good_target_mode": item["good_target_mode"],
            "bad_entropy": item["bad_entropy"],
            "good_entropy": item["good_entropy"],
            "sample_dir": item["sample_dir"],
        }
        for item in render_results
    ]
)
display(render_summary)
render_summary.to_csv(OUTPUT_ROOT / "render_summary.csv", index=False)


Render Gap-CAM samples: 100%|██████████| 8/8 [00:02<00:00,  3.52it/s]


,rank,label_name,bad_score,good_score,good_target_mode,bad_entropy,good_entropy,sample_dir
0,1,"washbasin, handbasin, washbowl, lavabo, wash-h...",0.433592,0.000575,positive_fallback,10.101959,10.726038,/mnt/media/emanuele/few_dimensions/analysis_2/...
1,2,chest,0.421999,0.000681,positive_fallback,10.558110,10.686795,/mnt/media/emanuele/few_dimensions/analysis_2/...
2,3,"microwave, microwave oven",0.413838,0.000702,positive_fallback,10.483161,10.631439,/mnt/media/emanuele/few_dimensions/analysis_2/...
3,4,"medicine chest, medicine cabinet",0.412021,0.000647,positive_fallback,10.713146,10.753374,/mnt/media/emanuele/few_dimensions/analysis_2/...
4,5,table lamp,0.407102,0.000637,positive_fallback,10.624430,10.699807,/mnt/media/emanuele/few_dimensions/analysis_2/...
5,6,"home theater, home theatre",0.405908,0.000618,positive_fallback,10.652898,10.673939,/mnt/media/emanuele/few_dimensions/analysis_2/...
6,7,soap dispenser,0.396671,0.000621,positive_fallback,10.620267,10.547679,/mnt/media/emanuele/few_dimensions/analysis_2/...
7,8,iPod,0.396144,0.000530,positive_fallback,10.634515,10.653142,/mnt/media/emanuele/few_dimensions/analysis_2/...


In [12]:
if "render_results" not in globals() or not render_results:
    raise RuntimeError("Run the rendering cell first to build aggregate summaries.")

per_sample_summary, category_summary = aggregate_render_results(render_results, output_dir=AGGREGATES_DIR)
display(per_sample_summary)
display(category_summary)


,rank,img_id,label_name,bad_score,good_score,good_target_mode,bad_entropy,good_entropy,sample_dir
0,1,293390,"washbasin, handbasin, washbowl, lavabo, wash-h...",0.433592,0.000575,positive_fallback,10.101959,10.726038,/mnt/media/emanuele/few_dimensions/analysis_2/...
1,2,534664,chest,0.421999,0.000681,positive_fallback,10.558110,10.686795,/mnt/media/emanuele/few_dimensions/analysis_2/...
2,3,429598,"microwave, microwave oven",0.413838,0.000702,positive_fallback,10.483161,10.631439,/mnt/media/emanuele/few_dimensions/analysis_2/...
3,4,489091,"medicine chest, medicine cabinet",0.412021,0.000647,positive_fallback,10.713146,10.753374,/mnt/media/emanuele/few_dimensions/analysis_2/...
4,5,356432,table lamp,0.407102,0.000637,positive_fallback,10.624430,10.699807,/mnt/media/emanuele/few_dimensions/analysis_2/...
5,6,232088,"home theater, home theatre",0.405908,0.000618,positive_fallback,10.652898,10.673939,/mnt/media/emanuele/few_dimensions/analysis_2/...
6,7,541291,soap dispenser,0.396671,0.000621,positive_fallback,10.620267,10.547679,/mnt/media/emanuele/few_dimensions/analysis_2/...
7,8,53909,iPod,0.396144,0.000530,positive_fallback,10.634515,10.653142,/mnt/media/emanuele/few_dimensions/analysis_2/...


,label_name,n_samples,mean_bad_entropy,mean_good_entropy
0,"washbasin, handbasin, washbowl, lavabo, wash-h...",1,10.101959,10.726038
2,"microwave, microwave oven",1,10.483161,10.631439
1,chest,1,10.558110,10.686795
6,soap dispenser,1,10.620267,10.547679
4,table lamp,1,10.624430,10.699807
7,iPod,1,10.634515,10.653142
5,"home theater, home theatre",1,10.652898,10.673939
3,"medicine chest, medicine cabinet",1,10.713146,10.753374
